<!-- # Pipeline smoke test

Raw Citi Bike data -> `RawModelData` -> `ResolvedModelData` -> `Environment` -> `SimulationLog`.

The base scenario reproduces historical **demand** exactly. `FormDeparturesPhase`
re-releases every historical departure (gated by stock), and
`FormPotentialTripsPhase` assigns each departure a destination and duration from
the OD demand model `P(target | source, commodity)` + mean historical duration.

To make the replay exact, the resolved data is built with `saturate_stock=True`:
artificial saturated stock and dock capacities replace the GBFS snapshot, so the
demand gate and the overflow redirect stay in the pipeline but never bind. (The
GBFS snapshot is a current observation, unrelated to the historical start state,
and would starve the replay with stockouts that never happened.)

Because targets and durations come from the (aggregate) OD model rather than each
trip's own record, the per-trip journal is **not** identical to history, and the
OD-count matrix drifts slightly under per-period largest-remainder rounding. The
marginal that is preserved exactly is the departures table:
`simulated_departures_df == historical_departures_df`. -->

In [12]:
import pandas as pd

from gbp.loaders.dataloader_raw import RawModelData
from gbp.loaders.dataloader_graph import ResolvedModelData, attach_simulation
from gbp.consumers.simulator.engine import Environment, EnvironmentConfig
from gbp.consumers.simulator import (
    DockArrivals,
    FormDeparturesPhase,
    FormPotentialTripsPhase,
)

In [14]:
ubuntu_path = "/mnt/outer/Documents/vlzm/GFDRR_ubuntu/GFDRR/data/raw/202602-citibike-tripdata_1.csv"
mac_path = "/Users/vladislav/Documents/vlzm/GFDRR/data/raw/202601-citibike-tripdata_1.csv"

# Raw model data: read the trip CSV and the live GBFS feed, derive raw entities.
raw_data = RawModelData(
    gbfs_base="https://gbfs.citibikenyc.com/gbfs/en",
    trips_path=mac_path,
    seed=42,
    n_depots=10,
    depot_capacity=9000,
    n_trucks=5,
    truck_capacity_bikes=20,
    truck_rate=50.0,
    electric_bike_rate=5,
    classic_bike_rate=3,
)

graph_data = ResolvedModelData(raw_data, period_len=pd.Timedelta(hours=1), scale_capacity_factor = 10)

historical_flows_df_raw = graph_data.historical_flows_df.copy()

/Users/vladislav/Documents/vlzm/GFDRR/gbp/loaders/dataloader_raw.py:221: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.stations_capacities_df['capacity'] = 100


In [ ]:
phases_canonical = [
    DockArrivals("previous"),
    FormDeparturesPhase(),
    FormPotentialTripsPhase(),
    DockArrivals("same"),
]

env_canonical = Environment(
    graph_data,
    EnvironmentConfig(phases=phases_canonical, 
                      seed=42, 
                      scenario_id="historical_replay", 
                      demand_scale_factor=1.0,
                      number_of_periods = 30),
)
env_canonical.run()

attach_simulation(graph_data, env_canonical.simulated_flows_df)
simulated_flows_df = graph_data.simulated_flows_df
simulated_departures_df = graph_data.simulated_departures_df
historical_departures_df = graph_data.historical_departures_df

In [ ]:
from gbp.loaders.dataloader_graph import get_flows_wide, slice_flows_wide

simulated_flows_wild_df = get_flows_wide(graph_data)
simulated_flows_wild_df

,flow_id,move_id,event_id,period_id,flow_type,event_type,commodity_category,source_id,planned_target_id,realized_target_id,...,realized_target_capacity_total,realized_target_capacity_per_commodity_cat,realized_target_lat,realized_target_lng,realized_target_inventory_after,realized_target_inventory_before,planned_duration,realized_duration,planned_distance_km,realized_distance_km
0,sim_0_0,0,0,0,user_trip,departed,classic_bike,6626.01,5703.13,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,17,<NA>,3.206291,NaN
1,sim_4_0,0,0,4,user_trip,departed,classic_bike,6224.06,6339.06,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,24,<NA>,0.300107,NaN
2,sim_4_1,0,0,4,user_trip,departed,classic_bike,6257.06,6339.06,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,24,<NA>,0.575615,NaN
3,sim_4_2,0,0,4,user_trip,departed,classic_bike,7599.09,7599.02,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,25,<NA>,0.243180,NaN
4,sim_5_0,0,0,5,user_trip,departed,classic_bike,6030.04,6339.06,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,23,<NA>,1.115145,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30217,sim_29_998,0,0,29,user_trip,departed,electric_bike,6025.08,6932.14,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,0,<NA>,4.619503,NaN
30218,sim_29_998,0,1,29,user_trip,arrived,electric_bike,6025.08,6932.14,6932.14,...,1000.0,1000.0,40.767801,-73.965921,23,22,0,0,4.619503,4.619503
30219,sim_29_999,0,0,29,user_trip,departed,classic_bike,6030.04,6115.09,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,0,<NA>,0.838882,NaN
30220,sim_29_999,0,1,29,user_trip,arrived,classic_bike,6030.04,6115.09,6115.09,...,1000.0,1000.0,40.742754,-74.007474,20,18,0,0,0.838882,0.838882


In [ ]:
slice_flows_wide(simulated_flows_wild_df, "source", "6331.01", period_id=21, window=2)

,flow_id,move_id,event_id,period_id,flow_type,event_type,commodity_category,source_id,planned_target_id,realized_target_id,...,realized_target_capacity_total,realized_target_capacity_per_commodity_cat,realized_target_lat,realized_target_lng,realized_target_inventory_after,realized_target_inventory_before,planned_duration,realized_duration,planned_distance_km,realized_distance_km
11322,sim_18_662,0,1,19,user_trip,arrived,electric_bike,6331.01,6331.01,6331.01,...,1000.0,1000.0,40.749156,-73.991600,51,49,1,1,0.000000,0.000000
11846,sim_19_333,0,0,19,user_trip,departed,classic_bike,6331.01,6079.03,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,1,<NA>,1.614602,NaN
12287,sim_19_333,0,1,20,user_trip,arrived,classic_bike,6331.01,6079.03,6079.03,...,1000.0,1000.0,40.741444,-73.975361,19,17,1,1,1.614602,1.614602
13033,sim_21_165,0,0,21,user_trip,departed,electric_bike,6331.01,6079.03,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,0,<NA>,1.614602,NaN
13034,sim_21_165,0,1,21,user_trip,arrived,electric_bike,6331.01,6079.03,6079.03,...,1000.0,1000.0,40.741444,-73.975361,37,33,0,0,1.614602,1.614602
13035,sim_21_166,0,0,21,user_trip,departed,electric_bike,6331.01,6089.11,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,0,<NA>,1.262667,NaN
13036,sim_21_166,0,1,21,user_trip,arrived,electric_bike,6331.01,6089.11,6089.11,...,1000.0,1000.0,40.740693,-73.981606,27,25,0,0,1.262667,1.262667
13037,sim_21_167,0,0,21,user_trip,departed,electric_bike,6331.01,6089.11,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,0,<NA>,1.262667,NaN
13038,sim_21_167,0,1,21,user_trip,arrived,electric_bike,6331.01,6089.11,6089.11,...,1000.0,1000.0,40.740693,-73.981606,27,25,0,0,1.262667,1.262667
13571,sim_22_181,0,0,22,user_trip,departed,electric_bike,6331.01,6626.11,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,0,<NA>,1.411290,NaN


In [ ]:
slice_flows_wide(simulated_flows_wild_df, "planned_target",  "5294.04", period_id=12, window=2)

,flow_id,move_id,event_id,period_id,flow_type,event_type,commodity_category,source_id,planned_target_id,realized_target_id,...,realized_target_capacity_total,realized_target_capacity_per_commodity_cat,realized_target_lat,realized_target_lng,realized_target_inventory_after,realized_target_inventory_before,planned_duration,realized_duration,planned_distance_km,realized_distance_km


In [ ]:
slice_flows_wide(simulated_flows_wild_df, "realized_target", "6233.04", period_id=20, window=0)

,flow_id,move_id,event_id,period_id,flow_type,event_type,commodity_category,source_id,planned_target_id,realized_target_id,...,realized_target_capacity_total,realized_target_capacity_per_commodity_cat,realized_target_lat,realized_target_lng,realized_target_inventory_after,realized_target_inventory_before,planned_duration,realized_duration,planned_distance_km,realized_distance_km
